# Domain 2: Tool Design & MCP Integration
## Task Statement 2.1: Design Effective Tool Interfaces with Clear Descriptions and Boundaries
**Certification Weight**: 20% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how Claude uses JSON Schema tool descriptions as an semantic router, how to define explicit operational boundaries and negative guidance ("When NOT to use"), and configure `tool_choice` for deterministic sequencing.

---

## 🏗️ Architecture: Tool Disambiguation & Routing Flow

```mermaid
flowchart TD
    UserQuery["User Input:<br>'Where is my recent shipment?'"] --> Claude["Claude Tool Router<br>(claude-sonnet-4-5)"]
    
    Claude --> EvaluateDescriptions{"Evaluate JSON Schema<br>Descriptions & Boundaries"}
    
    EvaluateDescriptions -->|"Explicit Match"| ToolA["track_order(order_id)<br>Boundary: Active tracking numbers only"]
    EvaluateDescriptions -->|"Explicit Exclusion"| ToolB["get_customer_profile(customer_id)<br>Boundary: DO NOT use for package tracking"]
    
    ToolA --> Dispatch["Execute Tool Call"]
    Dispatch --> ToolResult["Return tool_result Block"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Tool Design Element | Best Practice Implementation | Exam Trap to Reject |
|---|---|---|
| **Negative Guidance** | Explicitly state what the tool should **not** be used for (e.g. "Do NOT call this tool for tracking numbers; use `track_order` instead"). | **TRAP**: Relying on vague names alone (e.g. `lookup_info`) without negative boundaries, causing frequent tool collision and misrouting. |
| **Parameter Constraints** | Specify concrete formats, regex, enums, and required flags in JSON Schema properties (e.g. `pattern: "^ORD-[0-9]{5}$"`). | **TRAP**: Passing open-ended `string` fields without constraints or validation rules. |
| **Deterministic Sequencing** | Use forced tool choice `tool_choice: {"type": "tool", "name": "verify_identity"}` on turn 1 to enforce sequence. | **TRAP**: Hoping the model chooses verification first via prompt instructions while keeping `tool_choice: "auto"`. |

---

## 💡 Architectural Decision Matrix

| Tool Configuration | Behavior | Best Use Case |
|---|---|---|
| `tool_choice: "auto"` | Model decides between tool call or text | General conversational agents and final resolution turns. |
| `tool_choice: "any"` | Model must invoke at least one tool | Exploration passes where text-only output is not permitted. |
| `tool_choice: {"type": "tool", "name": "..."}` | Model is strictly forced to run this specific tool | Turn 1 authentication, compliance gating, or mandatory telemetry capture. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.1 Tool Interface Design.")

### 🛠️ Step 2: Differentiated Tool Declarations with Boundaries
Notice the explicit `BOUNDARY:` directives in both tool descriptions to eliminate misrouting.

In [ ]:
# Step 2: Production Tool Schemas
TOOLS = [
    {
        "name": "get_customer",
        "description": (
            "Retrieves customer identity, contact information, and account status by email address or customer ID. "
            "Use this tool EXCLUSIVELY for verifying user identity, checking membership tier, or retrieving account profiles. "
            "BOUNDARY: Do NOT call this tool for order tracking, tracking numbers, or shipment history; use lookup_order instead. "
            "Example queries: 'Verify customer alice@example.com', 'Retrieve account profile for CUST-101'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "identifier": {
                    "type": "string",
                    "description": "Customer email (e.g. 'user@example.com') or Customer ID matching format 'CUST-XXXX'.",
                }
            },
            "required": ["identifier"],
        },
    },
    {
        "name": "lookup_order",
        "description": (
            "Retrieves order status, line items, shipment carrier tracking, and return eligibility for a specific order. "
            "Use this tool when users ask about delivery status, order contents, package location, or return windows. "
            "BOUNDARY: Do NOT call this tool for general user profile lookups without an order reference; use get_customer instead. "
            "Example queries: 'Where is order ORD-9821?', 'Check return eligibility for ORD-4412'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Order identifier matching format 'ORD-XXXX'.",
                }
            },
            "required": ["order_id"],
        },
    },
]

# Mock Database
CUSTOMER_DB = {"alice@example.com": {"customer_id": "CUST-101", "name": "Alice Smith", "tier": "Gold"}}
ORDER_DB = {"ORD-9821": {"order_id": "ORD-9821", "customer_id": "CUST-101", "carrier": "FedEx", "status": "In Transit"}}

def dispatch_tool(tool_name: str, inputs: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name == "get_customer":
        ident = inputs.get("identifier", "").strip().lower()
        cust = CUSTOMER_DB.get(ident)
        return {"found": True, "customer": cust} if cust else {"found": False, "isError": True, "errorCategory": "validation"}
    elif tool_name == "lookup_order":
        oid = inputs.get("order_id", "").strip().upper()
        ord_rec = ORDER_DB.get(oid)
        return {"found": True, "order": ord_rec} if ord_rec else {"found": False, "isError": True, "errorCategory": "validation"}
    return {"isError": True, "message": f"Unknown tool: {tool_name}"}

print(" Tool schemas and mock dispatch engine ready.")

### 🧪 Step 3: Interactive Simulation
We demonstrate how negative boundaries and forced tool choices direct execution cleanly.

In [ ]:
# Step 3: Simulation
# Test 1: Customer identity retrieval
res_cust = dispatch_tool("get_customer", {"identifier": "alice@example.com"})
print("Customer Lookup Result:", res_cust)

# Test 2: Order tracking retrieval
res_order = dispatch_tool("lookup_order", {"order_id": "ORD-9821"})
print("Order Lookup Result:", res_order)

# Test 3: Forced tool choice representation
forced_choice = {"type": "tool", "name": "get_customer"}
print("\nForced Tool Choice configuration payload:", forced_choice)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Both tool descriptions contain explicit negative boundary instructions.
2. Parameter descriptions specify regex/format conventions.
3. Both tool lookups retrieve accurate data.

In [ ]:
# Step 4: Verification Checks
get_cust_desc = next(t["description"] for t in TOOLS if t["name"] == "get_customer")
lookup_ord_desc = next(t["description"] for t in TOOLS if t["name"] == "lookup_order")

assert "BOUNDARY: Do NOT call this tool for order tracking" in get_cust_desc
assert "BOUNDARY: Do NOT call this tool for general user profile lookups" in lookup_ord_desc
assert "CUST-XXXX" in get_cust_desc or "CUST-XXXX" in str(TOOLS)
assert "ORD-XXXX" in lookup_ord_desc or "ORD-XXXX" in str(TOOLS)

assert res_cust["found"] is True
assert res_cust["customer"]["customer_id"] == "CUST-101"
assert res_order["order"]["carrier"] == "FedEx"

print(" All verification assertions passed! Tool interfaces verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Tool descriptions must define explicit **negative boundaries** (`BOUNDARY: Do NOT call this tool when...`).
- **Rule 2**: Specify exact parameter format constraints (`CUST-XXXX`) and provide concrete query examples.
- **Rule 3**: Use forced `tool_choice` (`{"type": "tool", "name": "..."}`) when a step is non-negotiable.